# Simone - AI Music Server

**使用方法：**
1. 确保运行时类型为 **A100 GPU**（菜单 → 运行时 → 更改运行时类型）
2. 按顺序运行 Cell 1 → Cell 2 → Cell 3
3. Cell 3 会输出 `wss://xxx.trycloudflare.com` 地址
4. 复制地址，粘贴到 Simone 前端的 `.env.local` 的 `NEXT_PUBLIC_WS_URL`

**重启服务（换 URL）：** 只需重跑 Cell 3，不用重启 Runtime

In [ ]:
# ====== Cell 1: 安装依赖（只需运行一次，约3-5分钟）======
!git clone https://github.com/magenta/magenta-realtime.git
%cd magenta-realtime

# 安装 T5X (GPU)
!git clone https://github.com/google-research/t5x.git && \
  cd t5x && git checkout 7781d16 && \
  patch setup.py < ../patch/t5x_setup.py.patch && \
  patch t5x/partitioning.py < ../patch/t5x_partitioning.py.patch && \
  pip install .[gpu] && cd ..

# 安装 Magenta RT
!pip install -e .[gpu] && pip install tf2jax==0.3.8

# 打 seqio patch
!patch $(python -c "import seqio; print(seqio.__file__.replace('__init__.py',''))"+'vocabularies.py') \
  < patch/seqio_vocabularies.py.patch || echo 'patch already applied or not needed'

# 额外依赖
!pip install websockets nest_asyncio

print('\n✅ 依赖安装完成！运行 Cell 2 加载模型。')

In [ ]:
# ====== Cell 2: 加载模型（只需运行一次，约60-80秒）======
import os
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

from magenta_rt import system
if 'mrt' not in dir() or mrt is None:
    print('Loading model...')
    mrt = system.MagentaRT()
    print('Model loaded. Warmup...')
    style0 = mrt.embed_style('chill ambient music')
    c0, s0 = mrt.generate_chunk(style=style0)
    print(f'Warmup done. Chunk shape: {c0.samples.shape}')
else:
    print('Model already loaded, skipping warmup.')
print('\n✅ 模型就绪！运行 Cell 3 启动服务器。')

In [ ]:
# ====== Cell 3: 从 GitHub 拉取最新 server 代码并启动 ======
# 每次运行都从 repo 拉最新版本，保证和前端代码同步
# 复用 Cell 2 已加载的 mrt 模型，无需重新加载
import urllib.request
server_url = 'https://raw.githubusercontent.com/Oldfishermannn/DJ-LaoYu/main/colab_server.py'
print('Fetching colab_server.py from GitHub...')
resp = urllib.request.urlopen(server_url)
server_code = resp.read().decode('utf-8')
print(f'Got {len(server_code)} bytes, starting server...\n')
compiled = compile(server_code, 'colab_server.py', 'exec')
run_globals = dict(globals())
run_globals['mrt'] = mrt
run_globals['__name__'] = '__main__'
exec(compiled, run_globals)  # noqa: S102 - intentional: loads trusted server code from our own repo